# Why basic RAG retrieval fails - and how reranking fixes it

A vector store finds a broad semantic neighborhood, but its top results are not always the best evidence for an exact question. This notebook builds a small research-paper-style corpus, runs ordinary dense retrieval, and reranks the same candidates with a cross-encoder.

The production pattern is **retrieve broadly, rerank narrowly**: retrieve enough candidates for recall, score each question/chunk pair, and send only the reranked results to generation. This notebook needs no API key or PDF files.


## 1. Setup

The first run may download the local embedding and reranking model weights.

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512)


## 2. Controlled corpus

Several chunks mention SciFact, transformers, evaluation, and F1, so dense similarity sees them as good neighbors. Only one chunk contains the exact result needed by the question. Metadata is retained for traceable citations.

In [ ]:
documents = [
    Document("Transformer baselines and BERT-style encoders are common in SciFact research. We compare pretrained language models, retrieval models, and evaluation settings for scientific claim verification.", metadata={"paper_id": "paper-a", "paper_title": "Evidence Retrieval Survey", "page": 2}),
    Document("On SciFact, we compare reports of an F1 score of 91.2% on the test set with BM25 and BERT retrieval baselines. This section discusses the evaluation wording but does not establish the proposed verifier result.", metadata={"paper_id": "paper-b", "paper_title": "Retrieval Baselines", "page": 5}),
    Document("The calibrated verifier's held-out benchmark score was 91.2 percent F1, exceeding every baseline in the final experiment.", metadata={"paper_id": "paper-c", "paper_title": "Calibrated Claim Verification", "page": 8}),
    Document("We evaluate transformer encoders on SciFact and discuss the reported 91.2% F1 score on the test set. BERT, RoBERTa, and DeBERTa are compared with dense retrieval and cross-encoder baselines.", metadata={"paper_id": "paper-d", "paper_title": "Scientific Text Encoders", "page": 6}),
    Document("The model is trained with contrastive learning on scientific claims. A transformer encoder creates representations for retrieval, while the final classifier predicts whether evidence supports the claim.", metadata={"paper_id": "paper-e", "paper_title": "Contrastive Evidence Models", "page": 4}),
    Document("Our ablation studies vary transformer depth, negative sampling, and the SciFact training split. Results are reported for precision, recall, F1, and mean reciprocal rank.", metadata={"paper_id": "paper-f", "paper_title": "Ablations for Scientific Retrieval", "page": 9}),
    Document("The evaluation protocol follows prior SciFact work and uses official evidence annotations. We report transformer baseline performance and analyze errors in retrieved scientific passages.", metadata={"paper_id": "paper-g", "paper_title": "Evidence Annotation Analysis", "page": 3}),
    Document("A cross-encoder improves ranking quality when several passages are semantically related. It jointly reads the question and candidate passage rather than comparing two independent embeddings.", metadata={"paper_id": "paper-h", "paper_title": "Reranking for Dense Retrieval", "page": 7}),
]
query = "Which paper reports an F1 score of 91.2% on the SciFact test set?"
answer_paper = "paper-c"
print(f"{len(documents)} chunks; answer paper is {answer_paper}")


## 3. Simple vector retrieval

We retrieve a broad candidate set but pass only the top two to the hypothetical answer generator. The first-stage retriever is optimized for speed and recall, not perfect ordering.

In [ ]:
store = Chroma(collection_name="reranking_demo", embedding_function=embeddings)
store.delete_collection()
store = Chroma.from_documents(documents, embedding=embeddings, collection_name="reranking_demo")
candidates = store.similarity_search_with_relevance_scores(query, k=len(documents))

def show(results, title):
    print(title)
    for rank, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        print(f"{rank:>2}. score={score:.4f} | {m['paper_id']} | {m['paper_title']} | page {m['page']}")
        print(f"    {doc.page_content}")

show(candidates[:2], "Simple RAG: dense top-2")
simple_papers = {doc.metadata["paper_id"] for doc, _ in candidates[:2]}
print("Answer paper retrieved:", answer_paper in simple_papers)


If the final line is `False`, simple RAG failed: related chunks crowded out the exact evidence. The result can vary slightly with embedding versions, which is precisely why ranking should be measured rather than assumed.

## 4. Cross-encoder reranking

A cross-encoder reads the question and each candidate together. It can recognize the exact relationship among `91.2%`, `F1`, `SciFact`, and `test set`. It is slower than vector search, so it is applied only to the small candidate set.

In [ ]:
pairs = [(query, doc.page_content) for doc, _ in candidates]
scores = reranker.predict(pairs)
reranked = sorted(
    [(doc, float(score)) for (doc, _), score in zip(candidates, scores)],
    key=lambda item: item[1],
    reverse=True,
)
show(reranked[:2], "Reranked RAG: cross-encoder top-2")
reranked_papers = {doc.metadata["paper_id"] for doc, _ in reranked[:2]}
print("Answer paper retrieved:", answer_paper in reranked_papers)


In [ ]:
print("| Pipeline | Answer paper found |")
print("|---|---|")
print(f"| Dense top-2 | {'PASS' if answer_paper in simple_papers else 'FAIL'} |")
print(f"| Dense top-8 + cross-encoder top-2 | {'PASS' if answer_paper in reranked_papers else 'FAIL'} |")

final_context = "\n\n".join(
    f"[{doc.metadata['paper_title']}, page {doc.metadata['page']}] {doc.page_content}"
    for doc, _ in reranked[:2]
)
print("\nContext to send to the answer model:\n")
print(final_context)


## 5. How to add this to the Docling RAG pipeline

Keep Docling, page metadata, Chroma filters, and incremental indexing unchanged. After `retrieve()` returns candidate `Document` objects, apply:

```python
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs, top_n=6):
    scores = reranker.predict([(query, doc.page_content) for doc in docs])
    ranked = sorted(zip(docs, scores), key=lambda item: float(item[1]), reverse=True)
    return [doc for doc, _ in ranked[:top_n]]
```

Use a larger first-stage `k` (for example 20-50), rerank it, then send only the final 4-8 chunks to the LLM. Keep `paper_id`, `paper_title`, `page`, and `chunk_id` unchanged so citations remain traceable. Reranking improves precision; it does not replace good chunking, metadata filters, or sufficient first-stage recall.